In [1]:
import time
import sqlite3
import requests
from bs4 import BeautifulSoup
import urllib3


urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


BANCO_DADOS = r"X:\KENEDY\banco_livros1.db"
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}


DELAY_REQUISICAO = 3.0  
MAX_VAZIOS_SEGUIDOS = 150 
ID_LIMITE_MAXIMO = 25000  

def inicializar_banco():
    conexao = sqlite3.connect(BANCO_DADOS)
    cursor = conexao.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS livros (
            id INTEGER PRIMARY KEY,
            titulo TEXT,
            autor TEXT,
            download TEXT,
            data_coleta TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conexao.commit()
    return conexao

def descobrir_proximo_id(conexao):
    cursor = conexao.cursor()
    cursor.execute('SELECT MAX(id) FROM livros')
    resultado = cursor.fetchone()
    if resultado[0] is None:
        return 10 
    return int(resultado[0]) + 1

def iniciar_varredura_total():
    conexao = inicializar_banco()
    cursor = conexao.cursor()
    
    id_atual = descobrir_proximo_id(conexao)
    vazios_seguidos = 0
    
    print(f"--- INICIANDO MOTOR DE VARREDURA COMPLETA ---")
    print(f"Retomando a partir do ID: {id_atual}\n")
    
    try:
        while True:
            if id_atual > ID_LIMITE_MAXIMO:
                print(f"\n[FIM DA LINHA] Limite máximo de ID ({ID_LIMITE_MAXIMO}) atingido!")
                break
                
            url = f"https://www.kufunda.net/pubview.php?id={id_atual}"
            
            resposta = None
            sucesso_conexao = False
            
            
            for tentativa in range(1, 4):
                try:
                    resposta = requests.get(url, headers=HEADERS, timeout=10, verify=False)
                    sucesso_conexao = True
                    break 
                except Exception as e:
                    print(f"[ID {id_atual}]  Falha na tentativa {tentativa}/3: Erro de Rede. Aguardando 5s...")
                    time.sleep(5)
            
            
            if not sucesso_conexao:
                print(f"[ID {id_atual}]  ID pulado por instabilidade persistente no servidor.")
                cursor.execute('INSERT OR IGNORE INTO livros (id, titulo, autor, download) VALUES (?, ?, ?, ?)', 
                               (id_atual, "Erro de Conexão/Bloqueio", "N/A"))
                conexao.commit()
                id_atual += 1
                continue
            
            
            if resposta.status_code == 200:
                soup = BeautifulSoup(resposta.text, 'html.parser')
                tag_titulo = soup.find('p', class_='title')
                tag_autor = soup.find('p', class_='author')
                tag_dow = soup.select_one('.doc-download a')
                if tag_titulo:
                    nome_livro = tag_titulo.get_text().strip()
                    nome_autor = tag_autor.get_text().strip() if tag_autor else "Autor Desconhecido"
                    nome_dow = tag_dow['href'].strip() if tag_dow else "Link Não Encontrado"
                    
                    cursor.execute('INSERT OR IGNORE INTO livros (id, titulo, autor, download) VALUES (?, ?, ?, ?)', (id_atual, nome_livro, nome_autor, nome_dow))
                    conexao.commit()
                    
                    print(f"[ID {id_atual}]  LIVRO MINERADO: {nome_livro}")
                    vazios_seguidos = 0  
                else:
                    vazios_seguidos += 1
                    print(f"[ID {id_atual}]  ID vago/vazio ({vazios_seguidos} seguidos)")
                    
            elif resposta.status_code == 404:
                vazios_seguidos += 1
                print(f"[ID {id_atual}]  Página 404 - Não existe ({vazios_seguidos} seguidos)")
            else:
                print(f"[ID {id_atual}]  Status inesperado do servidor: {resposta.status_code}")
            
            if vazios_seguidos >= MAX_VAZIOS_SEGUIDOS:
                print(f"\n[FIM AUTOMÁTICO] {MAX_VAZIOS_SEGUIDOS} páginas vazias seguidas. catálogo encerrado.")
                break
            
            id_atual += 1
            time.sleep(DELAY_REQUISICAO)
            
    except KeyboardInterrupt:
        print("\n[PAUSADO] Varredura interrompida pelo usuário.")
    finally:
        conexao.close()
        print("Banco de dados desconectado com segurança.")

if __name__ == "__main__":
    iniciar_varredura_total()


--- INICIANDO MOTOR DE VARREDURA COMPLETA ---
Retomando a partir do ID: 986

[ID 986]  LIVRO MINERADO: anatomia vegetal
[ID 987]  LIVRO MINERADO: Razão e liberdade em Leibniz
[ID 988]  LIVRO MINERADO: biologia celular
[ID 989]  LIVRO MINERADO: biologia botânica
[ID 990]  LIVRO MINERADO: biologia
[ID 991]  LIVRO MINERADO: Os movimentos sociais em Alain Touraine
[ID 992]  LIVRO MINERADO: EXERCÍCIOS DE VESTIBULAR
[ID 993]  LIVRO MINERADO: glossario plantas com flores
[ID 994]  Página 404 - Não existe (1 seguidos)
[ID 995]  LIVRO MINERADO: Célula – A Unidade da Vida
[ID 996]  LIVRO MINERADO: celula animal e vegetal 2
[ID 997]  LIVRO MINERADO: - CONSORCIAÇÃO MILHO-FEIJÃO
[ID 998]  LIVRO MINERADO: Morfologia externa de caules: organografia.
[ID 999]  LIVRO MINERADO: EVOLUÇÃO DAS PLANTAS NA NATUREZA
[ID 1000]  LIVRO MINERADO: BIOLOGIA – Transmissão da Vida
Exercícios complementares
Heranças sexuais e aneuploidia
[ID 1001]  LIVRO MINERADO: exemplo de relatorio musgo
[ID 1002]  LIVRO MINERADO: 